# Statistical Arbitrage Strategy: Intraday Mean-Reversion Pairs Trading
##### Yevgen Revtsov

## Strategy Summary

### Introduction

This research project demonstrates that cointegrated equity pairs within the same economic sector can be systematically exploited for profit through mean-reversion trading, using the Engle-Granger two-step methodology to identify stationary spreads and z-score normalization to generate entry and exit signals. First, we apply cointegration testing to within-sector stock pairs to identify relationships with long-run equilibrium properties, filtering by statistical significance (p-value ≤ 0.05) and mean-reversion speed. Second, we construct dollar-neutral spread positions using estimated hedge ratios, implementing strict risk management through stop losses and time limits to control tail risk and improve capital efficiency. Third, we backtest the strategy on 1-minute intraday data from 100 large-cap US equities across 11 GICS sectors, incorporating realistic transaction costs to validate the profitability of high-frequency statistical arbitrage. Fourth, we conduct comprehensive hypothesis testing across critical dimensions including pair selection methodology, signal generation parameters, transaction cost sensitivity, and out-of-sample robustness to establish the strategy's validity and identify optimization opportunities. Fifth, we perform walk-forward analysis and parameter sensitivity tests to ensure the strategy is not overfit to historical data and maintains consistent risk-adjusted returns across different market regimes. The goal is to show how modern cointegration-based pairs trading can generate positive risk-adjusted returns in liquid equity markets through systematic exploitation of temporary mispricings, while recognizing the critical importance of transaction cost management, robust pair selection, and disciplined risk controls for strategy success.

### Point 1: Cointegration-Based Pair Selection

The strategy employs the Engle-Granger two-step cointegration methodology to identify stock pairs with stationary spread relationships suitable for mean-reversion trading. The first step estimates the hedge ratio $\beta$ through ordinary least squares regression:

$$Y_t = \alpha + \beta \cdot X_t + \epsilon_t$$

where $Y_t$ and $X_t$ represent the price series of two stocks at time $t$. The second step tests the residuals (spread $Z_t = Y_t - \beta \cdot X_t$) for stationarity using the Augmented Dickey-Fuller (ADF) test, with the null hypothesis that the spread contains a unit root. Pairs passing the stationarity test at 95% confidence (p-value ≤ 0.05) and exhibiting half-lives between 5 and 120 observations are selected, with stronger preference given to lower p-values indicating more robust cointegration relationships. The restriction to within-sector pairs (using GICS sector classification) increases the probability of finding economically meaningful cointegration, as stocks in the same sector share common factor exposures including industry trends, commodity prices, regulatory changes, and macroeconomic forces. The key takeaway is that systematic cointegration testing with appropriate filters provides a statistically rigorous foundation for identifying pairs with predictable mean-reversion behavior, superior to correlation-based or distance-based methods that lack theoretical grounding in long-run equilibrium relationships.

### Point 2: Mean-Reversion Signal Generation via Z-Score Normalization

The strategy generates trading signals by normalizing the spread through z-score transformation, allowing standardized comparison across pairs with different volatility characteristics. The z-score at time $t$ is calculated as:

$$z_t = \frac{Z_t - \mu_{rolling}}{\sigma_{rolling}}$$

where $Z_t$ is the current spread value, $\mu_{rolling}$ is the rolling mean over a lookback window (default 60 observations), and $\sigma_{rolling}$ is the rolling standard deviation over the same window. Entry signals are generated when the z-score exceeds threshold levels: long spread entry at $z < -2.5$ (spread undervalued, expect reversion upward) and short spread entry at $z > +2.5$ (spread overvalued, expect reversion downward). Exit signals occur at mean reversion ($|z| \leq 0.0$), stop loss ($|z| > 4.0$ indicating relationship breakdown), or time limit (to prevent capital tie-up and overnight risk).

### Point 3: Market-Neutral Position Construction and Risk Management

The strategy constructs dollar-neutral positions at entry by taking offsetting positions in both legs of each pair, with position sizes determined by the hedge ratio to minimize directional market exposure. For a long spread position (when $z < -2.5$), the strategy buys \$1 of stock A and sells $\beta$ of stock B; for a short spread position (when $z > +2.5$), it sells \$1 of stock A and buys $\beta$ of stock B, where $\beta$ is the cointegration coefficient estimated from the regression. Capital allocation follows an equal-weight scheme with a maximum of 10 simultaneous pairs, each receiving \$10,000 from a \$100,000 portfolio, ensuring diversification across uncorrelated pair relationships. Risk management operates on three levels: (1) stop losses at $|z| = 4.0$ limit catastrophic losses from cointegration breakdown, (2) time stops at 120 minutes prevent capital from being tied up in slow-reverting positions and eliminate overnight gap risk, and (3) position limits ensure no overlapping trades in the same pair and bound maximum portfolio exposure. Transaction costs are modeled conservatively at 20 basis points per leg, encompassing bid-ask spreads, commissions, and market impact, totaling 80 basis points per round-trip trade (4 legs: buy A, sell B, exit A, exit B). The critical takeaway is that disciplined risk management and realistic cost assumptions are essential for translating theoretical statistical arbitrage edges into actual trading profits, as high-frequency strategies are particularly sensitive to execution quality and cost drag.

### Point 4: Intraday Implementation with Market Hours Filtering

The strategy operates on 1-minute intraday data with careful filtering to exclude after-hours trading and handle market holidays, ensuring that only regular trading hours (9:30 AM - 4:00 PM ET) are analyzed when liquidity and price discovery are optimal. Data preprocessing uses pandas_market_calendars with NYSE schedule to generate valid trading minute timestamps, filtering out pre-market (before 9:30 AM) and post-market (after 4:00 PM) periods where spreads widen and execution quality deteriorates. Half-day holiday detection identifies early market closes (1:00 PM). All timestamps are converted to US/Eastern timezone for consistency, and the data pipeline handles corporate actions (splits, dividends) through EODHD API integration. The universe consists of 100 large-cap and mid-cap stocks stratified across 11 GICS sectors, selected for high liquidity and sector diversity. The intraday timeframe provides two key advantages over daily data: (1) more frequent mean-reversion opportunities increase trade frequency and capital velocity, potentially improving risk-adjusted returns, and (2) faster position turnover reduces overnight risk and exposure to gap events from news announcements. The tradeoff is higher transaction cost sensitivity, making realistic cost modeling and execution quality paramount for strategy viability.

### Point 5: Hypothesis Testing and Validation Framework

The project establishes a comprehensive hypothesis testing framework covering high-priority dimensions to validate strategy assumptions, optimize parameters, and assess robustness to overfitting and regime changes. Core validation hypotheses (Phase 1) test whether within-sector pairs exhibit higher cointegration rates than random pairs (H1), whether optimal z-score entry thresholds exist (H2), whether the strategy remains profitable under realistic transaction cost assumptions (H3), and whether in-sample optimization leads to significant out-of-sample degradation (H4). Robustness hypotheses (Phase 2) examine the effectiveness of stop losses in reducing tail risk (H5), walk-forward consistency across rolling windows (H6), and whether lower ADF p-values predict higher profitability (H7) and compare performance across different timeframes from 1-minute to daily (H8). For each hypothesis, the testing methodology specifies the null hypothesis (H₀), alternative hypothesis (H₁), test procedure, evaluation metrics, and expected outcomes. The key goal is that systematic hypothesis testing transforms pairs trading from a heuristic approach into a rigorous quantitative framework, enabling evidence-based parameter selection and providing early warning signals for strategy degradation.

### Conclusion

This statistical arbitrage strategy synthesizes cointegration theory, mean-reversion dynamics, and disciplined risk management to systematically exploit temporary mispricings in equity pairs while maintaining market neutrality and controlling downside risk. The Engle-Granger methodology provides a statistically rigorous framework for pair selection based on long-run equilibrium relationships rather than spurious correlations, z-score normalization enables standardized signal generation across heterogeneous pairs with adaptive threshold setting, market-neutral positioning combined with stop losses and time limits ensures that profitable mean-reversion opportunities are captured while limiting exposure to relationship breakdowns and overnight risk, intraday implementation captures high-frequency inefficiencies while requiring careful attention to transaction costs and execution quality, and comprehensive hypothesis testing establishes empirical validity while guarding against overfitting and parameter instability. The strategy is relevant to quantitative portfolio managers and systematic traders seeking market-neutral alpha sources with limited directional exposure, particularly those with access to intraday data feeds, low-latency execution infrastructure. Success hinges on continuous monitoring of pair relationships for structural breaks, realistic modeling of transaction costs and slippage, disciplined adherence to risk management rules especially during market stress, and ongoing research into regime-dependent parameters and machine learning enhancements.

---

## Hypothesis Testing Framework

This section presents the hypotheses identified for testing, organized by priority phase. Each hypothesis includes the formal null and alternative hypotheses, testing methodology, evaluation metrics, and expected outcomes.

### Phase 1: Core Validation

#### Hypothesis 1: Within-Sector Cointegration

**Null Hypothesis (H₀)**: Stock pairs within the same economic sector are no more likely to be cointegrated than random pairs across sectors.

**Alternative Hypothesis (H₁)**: Within-sector pairs exhibit significantly higher cointegration rates than cross-sector pairs.

**Test Method**:
- Compare cointegration test pass rates (p < 0.05) for:
  - Within-sector pairs (stocks in same GICS sector)
  - Cross-sector random pairs (random pairing across different sectors)
- Use chi-square test for independence to determine if sector membership affects cointegration probability
- Calculate odds ratio to quantify the strength of the relationship
- Sample size: test all possible within-sector pairs vs. 1000 random cross-sector pairs

**Metrics**:
- Percentage of pairs passing ADF test at p < 0.05 (within-sector vs. cross-sector)
- Mean ADF statistic by pair type (more negative = stronger stationarity evidence)
- Mean p-value by pair type (lower = stronger cointegration)
- Odds ratio and 95% confidence interval
- Chi-square test statistic and p-value for independence test

**Expected Outcome**: Confirm H₁ (within-sector pairs significantly more cointegrated)

**Implications**:
- Validates sector-stratified universe design
- Justifies limiting computational search to within-sector pairs only
- Provides economic rationale (shared factor exposures) for statistical relationships
- If H₀ cannot be rejected: need to reconsider sector restriction or explore alternative grouping schemes (e.g., industry groups, factor-based clustering)

#### Hypothesis 2: Z-Score Entry Threshold Optimization

**Null Hypothesis (H₀)**: The choice of z-score entry threshold (e.g., 2.0 vs. 2.5 vs. 3.0) does not significantly affect risk-adjusted returns.

**Alternative Hypothesis (H₁)**: There exists an optimal z-score entry threshold that maximizes Sharpe ratio, balancing trade frequency and signal quality.

**Test Method**:
- Grid search over z_entry ∈ {1.5, 2.0, 2.5, 3.0, 3.5}
- For each threshold, run complete backtest on same data
- Calculate Sharpe ratio for each threshold
- Analyze tradeoff curves: Sharpe vs. threshold, frequency vs. threshold, win rate vs. threshold
- Perform sensitivity analysis: does optimal threshold vary by sector or pair characteristics?

**Metrics**:
- Sharpe ratio vs. z_entry (primary metric)
- Trade frequency vs. z_entry (number of trades per year)
- Win rate vs. z_entry (% of profitable trades)
- Average trade P&L vs. z_entry (mean profit per trade)
- Average holding time vs. z_entry
- Maximum drawdown vs. z_entry
- Bootstrap confidence intervals for Sharpe ratio at each threshold

**Expected Outcome**: Confirm H₁ (optimal threshold exists, likely around 2.0 - 2.5)

**Implications**:
- Higher threshold: fewer trades, higher win rate (stronger signals), lower frequency, potential underutilization of capital
- Lower threshold: more trades, lower win rate (noisier signals), higher frequency, higher transaction costs
- Tradeoff between signal quality and opportunity cost
- Optimal threshold may vary by volatility regime or pair characteristics
- If H₀ cannot be rejected: strategy may be robust to threshold choice, or Sharpe ratio may be poor metric (consider alternative objectives like Sortino, Calmar)

#### Hypothesis 3: Transaction Cost Sensitivity

**Null Hypothesis (H₀)**: Strategy remains profitable across a wide range of transaction cost assumptions (10-30 bps per leg).

**Alternative Hypothesis (H₁)**: Strategy is highly sensitive to transaction costs, with profitability disappearing above 25 bps per leg.

**Test Method**:
- Run backtests with transaction_cost_bps ∈ {5, 10, 15, 20, 25, 30}
- For each cost level, calculate complete performance metrics
- Identify breakeven cost level where Sharpe ratio = 0 (or returns = 0)
- Plot performance degradation curves
- Analyze which trade characteristics (holding time, z-score spread) are most affected by costs

**Metrics**:
- Sharpe ratio vs. transaction cost
- Annualized return vs. transaction cost
- Breakeven transaction cost (where returns = 0 or Sharpe = 0)
- Percentage of profitable trades vs. cost
- Profit factor (gross profit / gross loss) vs. cost
- Cost as % of gross P&L
- Sensitivity coefficient: ΔSharpe / Δcost

**Expected Outcome**: Confirm H₁ (costs are critical at high frequency)

**Implications**:
- Need highly accurate cost estimates for realistic performance projection
- Broker selection and negotiation crucial (sub-20 bps requires institutional access or high volume)
- May need to reduce trade frequency (higher z-score thresholds) if costs are elevated
- Execution quality (VWAP, TWAP algorithms, pairs algorithms) matters significantly
- If breakeven cost < 15 bps: strategy has limited practical viability without sophisticated execution
- If breakeven cost > 25 bps: strategy has cushion for retail implementation with modern zero-commission brokers
- Cost sensitivity likely higher at shorter timeframes (1-min) vs. longer (daily)

#### Hypothesis 4: In-Sample vs. Out-of-Sample Performance Degradation

**Null Hypothesis (H₀)**: In-sample optimized parameters perform equally well out-of-sample.

**Alternative Hypothesis (H₁)**: In-sample optimization leads to overfitting, with significant degradation in out-of-sample performance.

**Test Method**:
- Split data: 60% in-sample (IS), 40% out-of-sample (OOS)
- Optimize parameters on in-sample data (grid search for z_entry, zscore_window, stop loss)
- Test optimized parameters on held-out out-of-sample data
- Compare IS vs. OOS performance metrics
- Calculate degradation percentage = (Sharpe_IS - Sharpe_OOS) / Sharpe_IS
- Bootstrap confidence intervals for IS and OOS Sharpe ratios
- Test parameter stability: correlation between IS-optimal and OOS-optimal parameters

**Metrics**:
- Sharpe ratio: in-sample vs. out-of-sample
- Annualized return: IS vs. OOS
- Maximum drawdown: IS vs. OOS
- Win rate: IS vs. OOS
- Degradation percentage (for Sharpe, returns, win rate)
- Parameter stability: rank correlation of parameter performance IS vs. OOS
- Statistical significance of degradation (paired t-test on rolling window Sharpes)

**Expected Outcome**: Confirm H₁ (some degradation expected, but < 30%)

**Implications**:
- Expect 20-30% Sharpe degradation OOS as reasonable benchmark (per academic literature)
- Degradation > 50%: severe overfitting, parameters not robust
- Need conservative parameter selection: prefer robust over optimal
- Walk-forward analysis critical for realistic performance estimation
- If H₀ confirmed (no degradation): either parameters truly robust or insufficient difference between IS and OOS periods (regime stability)
- Result informs confidence in forward performance projections

### Phase 2: Robustness Testing

#### Hypothesis 5: Stop Loss Effectiveness for Tail Risk Reduction

**Null Hypothesis (H₀)**: A stop loss at z = 4.0 does not improve risk-adjusted returns compared to no stop loss.

**Alternative Hypothesis (H₁)**: The stop loss significantly reduces tail risk and improves Sharpe ratio by preventing catastrophic losses from cointegration breakdown.

**Test Method**:
- Compare backtest results with three configurations:
  1. With stop loss (z_stop = 4.0) - baseline
  2. Without stop loss (z_stop = ∞) - let positions run
  3. Alternative stop levels (z_stop ∈ {3.0, 3.5, 4.5, 5.0}) - sensitivity analysis
- Analyze drawdown and tail risk metrics
- Calculate percentage of trades stopped out vs. mean-reverted
- Analyze P&L distribution: compare left tail (5th percentile) with and without stops
- Test during crisis periods specifically (identify regime breaks)

**Metrics**:
- Maximum drawdown (with vs. without stop loss)
- 95th percentile loss (Value at Risk)
- Conditional Value at Risk (CVaR, expected loss beyond VaR)
- Sharpe ratio (risk-adjusted returns)
- Sortino ratio (downside deviation)
- Percentage of trades stopped out vs. mean-reverted
- Average P&L of stopped trades vs. mean-reverted trades
- Tail ratio (95th percentile gain / 5th percentile loss)

**Expected Outcome**: Confirm H₁ (stop loss improves risk-adjusted returns significantly)

**Implications**:
- Stop loss critical for risk management, prevents catastrophic losses from permanent relationship breakdowns
- Stopped-out trades indicate pair quality deterioration; should trigger pair re-evaluation
- Optimal stop level balances false positives (premature exit before reversion) vs. false negatives (letting losers run)
- Tighter stops (z = 3.0): lower drawdown but higher false positive rate, may reduce profitability
- Wider stops (z = 5.0): capture more reversions but allow larger drawdowns
- If H₀ confirmed: either pairs are very stable (rare stops triggered) or stop level is suboptimal (too wide or too tight)
- Result informs whether to implement pair-specific stops based on volatility or half-life

#### Hypothesis 6: Walk-Forward Robustness

**Null Hypothesis (H₀)**: Strategy performance is unstable across rolling walk-forward windows (high variance in out-of-sample Sharpe ratios).

**Alternative Hypothesis (H₁)**: Strategy delivers consistent risk-adjusted returns across multiple walk-forward periods, indicating genuine alpha rather than data mining.

**Test Method**:
- Use anchored or rolling walk-forward methodology:
  - 6-month in-sample period for parameter optimization
  - 3-month out-of-sample holdout for testing
  - Roll forward by 3 months, repeat
- For each OOS period, calculate Sharpe ratio using IS-optimized parameters
- Test consistency: percentage of periods with Sharpe > 1.0 (or > 0)
- Calculate mean and standard deviation of OOS Sharpe ratios
- Plot OOS Sharpe over time to identify regime changes
- Compare anchored vs. rolling walk-forward (does recency help?)

**Metrics**:
- Mean out-of-sample Sharpe ratio across all periods
- Standard deviation of OOS Sharpe ratios (lower = more consistent)
- Percentage of OOS periods with positive Sharpe
- Percentage of OOS periods with Sharpe > 1.0
- Worst OOS period Sharpe (downside risk)
- Information ratio (mean OOS Sharpe / std dev OOS Sharpe)
- Time series plot of OOS Sharpe ratios

**Expected Outcome**: Uncertain (critical robustness test)

**Implications**:
- If H₁ confirmed (consistent performance): strategy has genuine edge, not overfit to specific period
- If H₀ confirmed (inconsistent): strategy is curve-fit to historical data, parameters lack predictive power
- Low variance in OOS Sharpe: robust strategy across regimes
- High variance: performance highly regime-dependent, need regime filters or adaptive parameters
- Percentage of positive periods should be > 60% for viable strategy
- Negative periods should cluster (regime changes) rather than scatter randomly
- Result is most important validation: determines confidence in forward deployment

#### Hypothesis 7: P-Value as Pair Quality Signal

**Null Hypothesis (H₀)**: The p-value from the ADF cointegration test does not predict future pair profitability.

**Alternative Hypothesis (H₁)**: Pairs with lower p-values (stronger statistical significance of cointegration) generate higher risk-adjusted returns in backtesting.

**Test Method**:
- Sort all cointegrated pairs into quintiles by ADF test p-value (Q1 = lowest p-values, strongest cointegration)
- Run separate backtests for each quintile portfolio
- Compare Sharpe ratios across quintiles
- Analyze other metrics (win rate, average P&L, cointegration persistence) by quintile
- Statistical test: regression of pair Sharpe on p-value (or rank)

**Metrics**:
- Sharpe ratio by p-value quintile (expect decreasing with higher p-value)
- Average trade P&L by quintile
- Win rate by quintile
- Half-life by quintile (expect shorter for lower p-values)
- Percentage of pairs maintaining cointegration OOS by quintile
- Regression coefficient: Sharpe vs. p-value

**Expected Outcome**: Confirm H₁ (lower p-value → better performance)

**Implications**:
- Justifies focusing on top-ranked pairs (lowest p-values)
- Informs max_pairs selection: quality over quantity
- P-value threshold of 0.05 may be too lenient; consider 0.01 or 0.001 for stricter filtering
- If H₁ confirmed: can use p-value for pair weighting (allocate more capital to low p-value pairs)
- If H₀ confirmed: p-value is purely statistical artifact, doesn't predict trading profitability; need alternative quality metrics (half-life, R², variance ratio test)
- Result informs pair ranking and selection algorithm